# Test the Deployed Function App

Use this notebook to call the deployed Azure Function App HTTP starter and inspect the durable orchestration response. Update the endpoint and payload values before running the code cells.

In [23]:
import json
import os
import requests
import time

FUNCTION_APP_URL = os.environ.get("FUNCTION_APP_URL", "https://ragnonprod-gateway-func-gfftckaxapcpaeg3.westeurope-01.azurewebsites.net/api/orchestrators/start")
POLICY_PATH = os.environ.get("ODRL_POLICY_PATH", "../odrl_policies/20-privacy-analyst.json")
COSMOS_COLLECTION_ID = os.environ.get("COSMOS_COLLECTION_ID", "EnronEmailVectorStore")
QUERY_TEXT = os.environ.get("QUERY_TEXT", "Summarise the policy-compliant information about this topic.")

with open(POLICY_PATH, "r", encoding="utf-8") as policy_file:
    odrl_policy = json.load(policy_file)
payload = {
    "principal": {
        "userId": "notebook-user",
        "role": "privacy-analyst",
        "declaredIntent": "compliance_review",
    },
    "odrl_policy": odrl_policy,
    "query_text": QUERY_TEXT,
    "query_embedding": [0.1, 0.2, 0.3],
    "action": "summarise",
    "cosmos_collection": COSMOS_COLLECTION_ID
}

headers = {"Content-Type": "application/json"}
print("Starting policy-aware RAG request with payload:")
print(json.dumps(payload, indent=2))
print("Cosmos collection id:", COSMOS_COLLECTION_ID)
print("Query text:", QUERY_TEXT)




Starting policy-aware RAG request with payload:
{
  "principal": {
    "userId": "notebook-user",
    "role": "privacy-analyst",
    "declaredIntent": "compliance_review"
  },
  "odrl_policy": {
    "@context": "https://www.w3.org/ns/odrl.jsonld",
    "type": "Set",
    "uid": "policy:privacy-analyst",
    "profile": "PolicyAwareRAG base access policies",
    "rules": [
      {
        "uid": "rule:privacy-analyst:investigation",
        "action": [
          "summarise",
          "summarize",
          "retrieve",
          "audit",
          "redact"
        ],
        "assigner": "PolicyAwareRAG",
        "assignee": "privacy-analyst",
        "constraint": {
          "purpose": [
            "compliance_review",
            "fraud_detection",
            "security_review",
            "privacy_review"
          ]
        }
      }
    ]
  },
  "query_text": "Summarise the policy-compliant information about this topic.",
  "query_embedding": [
    0.1,
    0.2,
    0.3
  ],
  "act

In [24]:
payload = {
    "principal": {
        "userId": "notebook-user",
        "role": "privacy-analyst",
        "declaredIntent": "compliance_review"
    },
    "odrl_policy": odrl_policy,
    "query_text": QUERY_TEXT,
    "query_embedding": [0.1, 0.2, 0.3],
    "action": "summarise",
    "cosmos_collection": COSMOS_COLLECTION_ID
}

headers = {"Content-Type": "application/json"}
print("Using Cosmos collection id:", COSMOS_COLLECTION_ID)
print("Using query text:", QUERY_TEXT)
print(json.dumps(payload, indent=2))

Using Cosmos collection id: EnronEmailVectorStore
Using query text: Summarise the policy-compliant information about this topic.
{
  "principal": {
    "userId": "notebook-user",
    "role": "privacy-analyst",
    "declaredIntent": "compliance_review"
  },
  "odrl_policy": {
    "@context": "https://www.w3.org/ns/odrl.jsonld",
    "type": "Set",
    "uid": "policy:privacy-analyst",
    "profile": "PolicyAwareRAG base access policies",
    "rules": [
      {
        "uid": "rule:privacy-analyst:investigation",
        "action": [
          "summarise",
          "summarize",
          "retrieve",
          "audit",
          "redact"
        ],
        "assigner": "PolicyAwareRAG",
        "assignee": "privacy-analyst",
        "constraint": {
          "purpose": [
            "compliance_review",
            "fraud_detection",
            "security_review",
            "privacy_review"
          ]
        }
      }
    ]
  },
  "query_text": "Summarise the policy-compliant information

In [22]:
response = requests.post(FUNCTION_APP_URL, json=payload, headers=headers, timeout=60)
print("Start status:", response.status_code)

# Determine the status endpoint and poll until completion
status_url = None
try:
    body = response.json()
    status_url = body.get("statusQueryGetUri") or body.get("statusQueryGetUri".lower())
except Exception:
    pass

if not status_url:
    status_url = response.headers.get("Location")

if not status_url:
    print("Could not determine status URL; raw response follows:")
    print(response.text)
else:
    print("Polling orchestration status at:", status_url)
    while True:
        status_resp = requests.get(status_url)
        try:
            status_json = status_resp.json()
        except Exception:
            print("Non-JSON status response:", status_resp.text)
            break
        runtime = status_json.get("runtimeStatus")
        if runtime:
            print("runtimeStatus:", runtime)
        if runtime and runtime != "Running":
            print("Final orchestration output:")
            print(json.dumps(status_json.get("output"), indent=2))
            final_output = status_json.get("output")
            if isinstance(final_output, dict):
                print("Result status:", final_output.get("status"))
                if final_output.get("result") is not None:
                    print("Result:", final_output.get("result"))
                if final_output.get("reason") is not None:
                    print("Reason:", final_output.get("reason"))
            break
        time.sleep(1)


Start status: 202
Polling orchestration status at: https://ragnonprod-gateway-func-gfftckaxapcpaeg3.westeurope-01.azurewebsites.net/runtime/webhooks/durabletask/instances/530d1341857c4451a84f407ccd06a87e?taskHub=ragnonprodgatewayfunc&connection=Storage&code=qCO1YU6Jt9VHA6sQHXR9YB6EBz0eUgNvwr7Wbb86zmODAzFu9v5fhg==
runtimeStatus: Completed
Final orchestration output:
{
  "status": "ok",
  "result": "[LLM response placeholder]"
}
